# Melting Point 예측 — v2
데이터 정리 + 피처 확장 + 앙상블 (XGBoost · LightGBM · RF · Ridge)

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn xgboost lightgbm rdkit optuna shap -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, MACCSkeys
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator, GetRDKitFPGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold

import xgboost as xgb
import lightgbm as lgb
import optuna
import shap
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
print("라이브러리 로드 완료")

## 1. 데이터 로드 및 정리

In [ ]:
df = pd.read_csv("../data/Melting_point_2.csv")
print(f"원본 크기: {df.shape}")
print(f"결측치: {df.isnull().sum().sum()}")
print(f"중복 행: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"중복 제거 후: {df.shape}")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["MP"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Melting Point (K)")
axes[0].set_ylabel("Count")
axes[0].set_title("MP 분포 (정리 전)")
axes[1].boxplot(df["MP"])
axes[1].set_ylabel("Melting Point (K)")
axes[1].set_title("MP 박스플롯")
plt.tight_layout()
plt.show()
print(df["MP"].describe())

In [ ]:
# 음수 MP 확인 (절대온도 K 기준 물리적 불가)
neg_mask = df["MP"] < 0
print(f"음수 MP 샘플 수: {neg_mask.sum()}")
print(df[neg_mask])

# 음수 MP 제거
df = df[df["MP"] >= 0].reset_index(drop=True)
print(f"\n음수 제거 후: {df.shape}")
print(f"MP 범위: {df["MP"].min():.1f} ~ {df["MP"].max():.1f} K")

In [ ]:
# log1p 타깃 변환 (분포 균등화)
# log1p(x) = log(1+x) → 0에 가까운 값도 안전하게 처리
df["MP_log"] = np.log1p(df["MP"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["MP"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("원본 MP 분포")
axes[0].set_xlabel("MP (K)")
axes[1].hist(df["MP_log"], bins=50, color="salmon", edgecolor="white")
axes[1].set_title("log(MP) 분포 (변환 후)")
axes[1].set_xlabel("log1p(MP)")
plt.tight_layout()
plt.show()

## 2. 피처 엔지니어링

In [ ]:
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)
rdkit_gen  = GetRDKitFPGenerator(fpSize=2048)

# RDKit 기술자 200종 목록
desc_list = [(name, func) for name, func in Descriptors.descList
             if not name.startswith("Ipc")]
desc_names = [d[0] for d in desc_list]
print(f"RDKit 기술자 수: {len(desc_names)}")

def smiles_to_features(smiles_list):
    records, valid_idx = [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        # Morgan FP
        morgan = morgan_gen.GetFingerprintAsNumPy(mol).astype(np.float32)
        # RDKit FP
        rdkfp  = rdkit_gen.GetFingerprintAsNumPy(mol).astype(np.float32)
        # MACCS Keys
        maccs  = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
        # RDKit 기술자
        descs  = []
        for _, func in desc_list:
            try:
                v = func(mol)
                descs.append(float(v) if v is not None else 0.0)
            except:
                descs.append(0.0)
        descs = np.array(descs, dtype=np.float32)
        records.append(np.concatenate([morgan, rdkfp, maccs, descs]))
        valid_idx.append(i)
    return np.array(records), valid_idx

print("피처 추출 중... (시간 소요)")
X_all, valid_idx = smiles_to_features(df["SMILES"].tolist())
y_all     = df["MP"].values[valid_idx]
y_all_log = df["MP_log"].values[valid_idx]

total_dim = 2048 + 2048 + 167 + len(desc_names)
print(f"유효 샘플: {len(valid_idx)} / {len(df)}")
print(f"피처 차원: {X_all.shape[1]}  (Morgan {2048} + RDKit FP {2048} + MACCS {167} + 기술자 {len(desc_names)})")

## 3. 분산 필터링 (저분산 피처 제거)

In [ ]:
# 거의 변화가 없는 피처(분산 < 0.01) 제거
selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X_all)
print(f"필터링 전: {X_all.shape[1]}차원")
print(f"필터링 후: {X_filtered.shape[1]}차원 ({X_all.shape[1]-X_filtered.shape[1]}개 제거)")

In [ ]:
# 고MP 구간(상위 10%) 샘플에 3배 가중치 부여
high_mp_threshold = np.percentile(y_all, 90)
sample_weight = np.where(y_all >= high_mp_threshold, 3.0, 1.0)
print(f"고MP 기준값: {high_mp_threshold:.1f} K")
print(f"가중치 3배 샘플 수: {(sample_weight==3).sum()}")

## 4. Train / Test 분할

In [ ]:
from sklearn.model_selection import StratifiedKFold

# MP 구간 10개로 나눠 층화 분할
mp_bins = pd.qcut(y_all, q=10, labels=False, duplicates="drop")

X_train, X_test, y_train, y_test, y_train_log, y_test_log, sw_train, _ = train_test_split(
    X_filtered, y_all, y_all_log, sample_weight,
    test_size=0.2, random_state=RANDOM_STATE, stratify=mp_bins
)
print(f"Train: {X_train.shape[0]}개  |  Test: {X_test.shape[0]}개")

## 5. 하이퍼파라미터 튜닝 (Optuna)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
mp_bins_train = pd.qcut(y_train, q=10, labels=False, duplicates="drop")

def objective(trial):
    params = {
        "max_depth":             trial.suggest_int("max_depth", 2, 5),
        "min_child_weight":      trial.suggest_int("min_child_weight", 5, 30),
        "reg_alpha":             trial.suggest_float("reg_alpha", 1.0, 20.0, log=True),
        "reg_lambda":            trial.suggest_float("reg_lambda", 5.0, 50.0, log=True),
        "subsample":             trial.suggest_float("subsample", 0.4, 0.7),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.3, 0.5),
        "learning_rate":         trial.suggest_float("learning_rate", 0.005, 0.02, log=True),
        "n_estimators":          2000,
        "early_stopping_rounds": 50,
        "tree_method":           "hist",
        "random_state":          RANDOM_STATE,
        "n_jobs":                -1,
        "verbosity":             0,
    }
    scores = []
    for tr_idx, val_idx in skf.split(X_train, mp_bins_train):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train_log[tr_idx], y_train_log[val_idx]
        sw_tr = sw_train[tr_idx]
        m = xgb.XGBRegressor(**params)
        m.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_val, y_val)], verbose=False)
        pred_log = m.predict(X_val)
        pred = np.expm1(pred_log)
        scores.append(r2_score(np.expm1(y_val), pred))
    return float(np.mean(scores) - 0.5 * np.std(scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params.copy()
best_params.update({
    "n_estimators": 2000, "early_stopping_rounds": 50,
    "tree_method": "hist", "verbosity": 0,
    "n_jobs": -1, "random_state": RANDOM_STATE
})
print("Best score (mean-0.5*std):", round(study.best_value, 4))
print("Best params:", best_params)

## 6. 앙상블 모델 정의

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(**best_params)

# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=best_params["learning_rate"],
    max_depth=best_params["max_depth"], subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    reg_alpha=best_params["reg_alpha"], reg_lambda=best_params["reg_lambda"],
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)

# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=500, max_depth=best_params["max_depth"]+2,
    random_state=RANDOM_STATE, n_jobs=-1
)

# Ridge (이질적 모델 — 선형)
scaler = StandardScaler()
ridge_model = Ridge(alpha=10.0)

print("앙상블 모델 4개 정의 완료: XGBoost · LightGBM · RandomForest · Ridge")

## 7. 5-Fold Cross Validation (앙상블)

In [ ]:
cv_r2, cv_mse, cv_mae = [], [], []
oof_preds = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, mp_bins_train), 1):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr_log, y_val_log = y_train_log[tr_idx], y_train_log[val_idx]
    y_val_orig = y_train[val_idx]
    sw_tr = sw_train[tr_idx]

    # XGBoost
    xgb_model.fit(X_tr, y_tr_log, sample_weight=sw_tr,
                  eval_set=[(X_val, y_val_log)], verbose=False)
    p_xgb = np.expm1(xgb_model.predict(X_val))

    # LightGBM
    lgb_model.fit(X_tr, y_tr_log, sample_weight=sw_tr)
    p_lgb = np.expm1(lgb_model.predict(X_val))

    # Random Forest
    rf_model.fit(X_tr, y_tr_log, sample_weight=sw_tr)
    p_rf = np.expm1(rf_model.predict(X_val))

    # Ridge (StandardScaler 적용)
    X_tr_sc = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    ridge_model.fit(X_tr_sc, y_tr_log, sample_weight=sw_tr)
    p_ridge = np.expm1(ridge_model.predict(X_val_sc))

    # 가중 평균 (CV R² 기반)
    weights = np.array([
        max(r2_score(y_val_orig, p_xgb), 0),
        max(r2_score(y_val_orig, p_lgb), 0),
        max(r2_score(y_val_orig, p_rf),  0),
        max(r2_score(y_val_orig, p_ridge),0),
    ])
    weights = weights / weights.sum()
    pred = weights[0]*p_xgb + weights[1]*p_lgb + weights[2]*p_rf + weights[3]*p_ridge
    oof_preds[val_idx] = pred

    cv_r2.append(r2_score(y_val_orig, pred))
    cv_mse.append(mean_squared_error(y_val_orig, pred))
    cv_mae.append(mean_absolute_error(y_val_orig, pred))
    print(f"Fold {fold}  R²={cv_r2[-1]:.4f}  MSE={cv_mse[-1]:.2f}  MAE={cv_mae[-1]:.2f}  weights={weights.round(2)}")

print()
print("=== 5-Fold CV 평균 ===")
print(f"R²  : {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")
print(f"MSE : {np.mean(cv_mse):.2f} ± {np.std(cv_mse):.2f}")
print(f"MAE : {np.mean(cv_mae):.2f} ± {np.std(cv_mae):.2f}")

## 8. SHAP 기반 피처 선택 (Top 300)

In [ ]:
# 앙상블 완성 후 XGBoost SHAP으로 중요 피처 선별
X_tr_full, X_val_shap, y_tr_full_log, _ = train_test_split(
    X_train, y_train_log, test_size=0.2, random_state=RANDOM_STATE
)
xgb_shap = xgb.XGBRegressor(**best_params)
xgb_shap.fit(X_tr_full, y_tr_full_log,
             eval_set=[(X_val_shap, _)], verbose=False)

explainer = shap.TreeExplainer(xgb_shap)
shap_values = explainer.shap_values(X_train[:500])  # 샘플 500개로 계산
mean_shap = np.abs(shap_values).mean(axis=0)
top300_idx = np.argsort(mean_shap)[::-1][:300]

X_train_shap = X_train[:, top300_idx]
X_test_shap  = X_test[:, top300_idx]
print(f"SHAP top-300 피처 선별 완료: {X_train.shape[1]} → {X_train_shap.shape[1]}")

## 9. 최종 모델 — Test 평가

In [ ]:
# SHAP top-300 피처로 최종 학습
X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_train_shap, y_train_log, test_size=0.1, random_state=RANDOM_STATE
)

xgb_final = xgb.XGBRegressor(**best_params)
xgb_final.fit(X_tr_f, y_tr_f, eval_set=[(X_val_f, y_val_f)], verbose=False)

lgb_final = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=best_params["learning_rate"],
    max_depth=best_params["max_depth"], subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    reg_alpha=best_params["reg_alpha"], reg_lambda=best_params["reg_lambda"],
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)
lgb_final.fit(X_tr_f, y_tr_f)

rf_final = RandomForestRegressor(
    n_estimators=500, max_depth=best_params["max_depth"]+2,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_final.fit(X_tr_f, y_tr_f)

scaler_final = StandardScaler()
ridge_final = Ridge(alpha=10.0)
ridge_final.fit(scaler_final.fit_transform(X_tr_f), y_tr_f)

# 테스트 예측 (역변환)
p_xgb   = np.expm1(xgb_final.predict(X_test_shap))
p_lgb   = np.expm1(lgb_final.predict(X_test_shap))
p_rf    = np.expm1(rf_final.predict(X_test_shap))
p_ridge = np.expm1(ridge_final.predict(scaler_final.transform(X_test_shap)))

# 균등 가중 평균
y_pred = (p_xgb + p_lgb + p_rf + p_ridge) / 4

test_r2  = r2_score(y_test, y_pred)
test_mse = mean_squared_error(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)

print("=== Test Set 성능 ===")
print(f"R²  : {test_r2:.4f}")
print(f"MSE : {test_mse:.2f}")
print(f"MAE : {test_mae:.2f}")

## 10. 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.4, s=15, color="steelblue")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect")
ax.set_xlabel("실제 MP (K)")
ax.set_ylabel("예측 MP (K)")
ax.set_title(f"Predicted vs Actual  (R²={test_r2:.3f})")
ax.legend()

ax = axes[1]
residuals = y_pred - y_test
ax.hist(residuals, bins=40, color="salmon", edgecolor="white")
ax.axvline(0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("잔차 (예측 − 실제)")
ax.set_ylabel("Count")
ax.set_title("잔차 분포")

ax = axes[2]
ax.bar(range(1, 6), cv_r2, color="mediumseagreen", edgecolor="white")
ax.axhline(np.mean(cv_r2), color="red", linestyle="--",
           label=f"Mean={np.mean(cv_r2):.3f}")
ax.set_xlabel("Fold")
ax.set_ylabel("R²")
ax.set_title("5-Fold CV R² per Fold")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.show()

## 11. 결과 요약

In [ ]:
summary = pd.DataFrame({
    "구분":  ["5-Fold CV 평균", "Test Set"],
    "R²":   [f"{np.mean(cv_r2):.4f}", f"{test_r2:.4f}"],
    "MSE":  [f"{np.mean(cv_mse):.2f}", f"{test_mse:.2f}"],
    "MAE":  [f"{np.mean(cv_mae):.2f}", f"{test_mae:.2f}"],
})
print("=== base vs v2 비교 ===")
print(f"base  CV R²: 0.5346  Test R²: 0.4831")
print(f"v2    CV R²: {np.mean(cv_r2):.4f}  Test R²: {test_r2:.4f}")
summary